Quick summary / priorities

Structure: pivot ml_df to a wide country×year table (one row per country-year, columns = indicators).

Basic features: create lag, YoY, YoY_pct, rolling mean/std.

Spikes & dips: detect sudden jumps with threshold + z-score and add binary flags and magnitude features.

Volatility: rolling std of YoY or returns, plus coefficient of variation.

Meta features: region, subregion, income, source coverage, indicator counts.

Cleaning for ML: handle remaining NaNs (row/column thresholds, impute), scale, encode.

Dimensionality & selection: PCA/UMAP, correlation filtering, LASSO/Tree-based importance.

Modeling & evaluation: clustering/unsupervised first (profiles), then supervised tasks if you have labels. Use time-aware CV.

Explainability & diagnostics: SHAP, partial dependence, feature drift checks.

Packaging: pipeline objects, export CSVs/models, reproducible notebook.

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(os.getcwd())

/Users/daniehbenotman/Desktop/worldbank_tableau_project/notebooks


In [3]:
os.chdir("/Users/daniehbenotman/Desktop/worldbank_tableau_project")
print("✅ Working directory changed to:", os.getcwd())

✅ Working directory changed to: /Users/daniehbenotman/Desktop/worldbank_tableau_project


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
import numpy as np

In [5]:
fe_df = pd.read_csv("data/feature_data.csv")

In [6]:
fe_df.head()

,country_name,year,value,indicator_id,indicator_name,unit_type,source,source_reliable,region,category
0,Algeria,2000,0.0,SG.APL.PSPT.EQ,A woman can apply for a passport in the same w...,Binary (1/0),WHO,high,North Africa,Other
1,Algeria,2001,0.0,SG.APL.PSPT.EQ,A woman can apply for a passport in the same w...,Binary (1/0),WHO,high,North Africa,Other
2,Algeria,2002,0.0,SG.APL.PSPT.EQ,A woman can apply for a passport in the same w...,Binary (1/0),WHO,high,North Africa,Other
3,Algeria,2003,0.0,SG.APL.PSPT.EQ,A woman can apply for a passport in the same w...,Binary (1/0),WHO,high,North Africa,Other
4,Algeria,2004,0.0,SG.APL.PSPT.EQ,A woman can apply for a passport in the same w...,Binary (1/0),WHO,high,North Africa,Other


In [7]:
fe_df["indicator_name"].nunique()


42

In [8]:
fe_df["country_name"].nunique()

18

In [9]:
fe_df["year"].min()

2000

In [10]:
fe_df["year"].max()

2023

In [11]:
fe_df["value"].isna().sum()

0